# Étape 5 - Prévision et visualisation

On utilise le modèle entraîné pour prédire le chiffre d'affaires de la semaine à venir,
puis on trace le résultat.

## 1. Importer les librairies

In [ ]:
import sys
sys.path.append('..')
import yaml
import logging
import logging.config
import numpy as np
import pandas as pd
pd.set_option('display.min_rows', 500)
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 500)
pd.set_option('max_colwidth', 400)

from foodcast.domain.transform import etl
from foodcast.domain.feature_engineering import features_offline, features_online
from foodcast.domain.forecast import span_future, cross_validate, plotly_predictions
from foodcast.domain.multi_model import MultiModel
from sklearn.ensemble import RandomForestRegressor
import foodcast.settings as settings
import plotly.graph_objects as go

with open(settings.LOGGING_CONFIGURATION_FILE, 'r') as f:
    logging.config.dictConfig(yaml.safe_load(f.read()))

%load_ext autoreload
%autoreload 2

## 2. Reprendre les étapes 1 à 4

In [ ]:
# --- Reprise des étapes 1 à 4 ---
# jeu d'entraînement
df = etl(settings.DATA_DIR, 197, 200)
df = features_offline(df)
x_train = df.drop(columns=['cash_in']).set_index('order_date')
y_train = df[['order_date', 'cash_in']].set_index('order_date')['cash_in']

# jeu de prédiction
past = etl(settings.DATA_DIR, 200, 200)
future = span_future(past['order_date'].max())
future = features_online(future, past)
future = future.set_index('order_date')

# modèle simple entraîné sur tout
simple_model = RandomForestRegressor(n_estimators=10, random_state=42)
simple_model.fit(x_train, y_train)
future.head()

## 3. Prédire le chiffre d'affaires futur

`simple_model.predict(future)` renvoie un tableau numpy. On le range dans un `DataFrame`
avec **le même index que `future`** et une seule colonne `y_pred_simple`
(ce nom est celui attendu par `plotly_predictions`).

In [ ]:
y_pred = pd.DataFrame(
    simple_model.predict(future),
    index=future.index,
    columns=['y_pred_simple'],
)
y_pred.head(20)

## 4. Tracer la prévision

In [ ]:
plotly_predictions(y_pred)

On obtient **une seule courbe** de prévision. Elle reproduit bien les cycles
jour/nuit et semaine, mais ne dit rien sur la **confiance** qu'on peut lui accorder.

➡️ Étape suivante : `06_incertitudes_multimodel.ipynb`